# Lab 10

We've been using probability as a motivation for multidimensional integration.  An application in data science is natural language processing.  In this lab we will explore how a computer might determine which topic a given sentence is about.  Then we will also generate sentences.


In [4]:
from pathlib import Path

import nltk
import numpy as np
from nltk.tokenize import word_tokenize

# Give NLTK a private cache for its tokenizer and corpus data.
nltk_dir = Path.home() / ".cache" / "stor235" / "nltk_data"
nltk_dir.mkdir(mode=0o700, parents=True, exist_ok=True)
nltk.data.path.insert(0, str(nltk_dir))

def ensure_nltk_resource(resource, package):
    try:
        nltk.data.find(resource, paths=[str(nltk_dir)])
    except LookupError:
        nltk.download(package, download_dir=str(nltk_dir), raise_on_error=True)

ensure_nltk_resource("tokenizers/punkt_tab/english/", "punkt_tab")

Here is some modifications to the code we used in Lab 2 to parse strings of text into words and form vectors

In [5]:
def clean_string(str):
# Keep only alphabetical characters in the string
    cleaned_string = ""
    for char in str:
        if char.isalpha() or char==" ":  # Check if the character is a letter or a space
            cleaned_string += char
    return cleaned_string.lower()

def build_clean_set(*dicts):
# It creates a set of all the words that appear, then returns them in a list in alphabetical order.
    s = set() #makes empty set
    for d in dicts:
        for word in d.keys():
          s.add(word)
    return sorted(s)

def word_dictionary(str1):
#returns a dictonary containing all words in string
#e.g. str1 = 'quick brown fox is quick.'
# returns {quick:1, brown:1, fox:1, is:1}
  x = {}
  str1 = clean_string(str1)
  words = word_tokenize(str1)
  for b in words:
            x[b]=1
  return x

def word_count(str1, word_dict):
# word_dict: an already-build dictionary of words
# returns vector containing counts of str1 words based on dictionary
  x = {}
  for b in word_dict:
    x[b]=0
  str1 = clean_string(str1)
  words = word_tokenize(str1)
  for b in words:
    if b in x:
      x[b]+=1 # do noting if the word isn't in the dictionary
  return x

def word_vector(word_set, word_dict):
    vector = []
    for word in word_set:
        # Get the value from word_dict or use 0 if the word is not found
        value = word_dict.get(word, 0)
        vector.append(value)
    return vector

def make_dictionary(*strings):
  word_dicts = []
  for s in strings:
    word_dicts.append(word_dictionary(s))
  word_set = build_clean_set(*word_dicts)
  vectors = []
  for d in word_dicts:
    vectors.append(word_vector(word_set,d))
  # print(word_set)
  return word_set

The function `make_dictionary` when given a single string, returns a vector of sorted unique words appearing in that sentence.

In [6]:
s1 = "I love watching Alfred Hitchcock movies."
s2 = "I hate watching Alfred Hitchcock movies."
s3 = "Squirrels love to attack my bird feeder."

word_dict = make_dictionary(s1)
print(word_dict)

word_dict = make_dictionary(s1+" "+s2+" "+s3)
print(word_dict)


['alfred', 'hitchcock', 'i', 'love', 'movies', 'watching']
['alfred', 'attack', 'bird', 'feeder', 'hate', 'hitchcock', 'i', 'love', 'movies', 'my', 'squirrels', 'to', 'watching']


The function `word_count` takes a string and a dictionary,  returning a vector of word counts from the string

In [7]:
# make a dictionary over all the words you might see
word_dict = make_dictionary(s1+" "+s2+" "+s3)

# count how many words in one string
x1 = word_count(s1, word_dict)
x1 = np.array( list(x1.values()) )  # maybe you know a better way to extract the numbers
print(x1)


[1 0 0 0 0 1 1 1 1 0 0 0 1]


##Part I: Creating the probability "density" for different topics

I put density in quotes, becasue we are working with discrete numbers, not a continuum.

**Problem 1:** Compute the *probability* a word appears in a string.  You should have three probabilities for each word, one for each string.  Print these probabilities.

In [8]:
def count_vector(text, vocab):
    counts = word_count(text, vocab)
    return np.array([counts[word] for word in vocab])

def word_probabilities(text, vocab):
    counts = count_vector(text, vocab)
    return counts / counts.sum()

example_vocab = make_dictionary(s1, s2, s3)
example_probs = [
    word_probabilities(text, example_vocab)
    for text in (s1, s2, s3)
]

print(f"{'word':12} {'s1':>8} {'s2':>8} {'s3':>8}")
for word, p1, p2, p3 in zip(example_vocab, *example_probs):
    print(f"{word:12} {p1:8.6f} {p2:8.6f} {p3:8.6f}")

word               s1       s2       s3
alfred       0.166667 0.166667 0.000000
attack       0.000000 0.000000 0.142857
bird         0.000000 0.000000 0.142857
feeder       0.000000 0.000000 0.142857
hate         0.000000 0.166667 0.000000
hitchcock    0.166667 0.166667 0.000000
i            0.166667 0.166667 0.000000
love         0.166667 0.000000 0.142857
movies       0.166667 0.166667 0.000000
my           0.000000 0.000000 0.142857
squirrels    0.000000 0.000000 0.142857
to           0.000000 0.000000 0.142857
watching     0.166667 0.166667 0.000000


Now we want build probabilities words appear based on different topics.  I asked ChatGPT for one paragraph of text about these three topics:

In [9]:
# Sports:
sports = "Sports play a vital role in fostering teamwork, discipline, and resilience, both at the professional and recreational levels. They bring people together across cultures and generations, uniting communities around shared goals and healthy competition. From the precision of tennis and the strategy of soccer to the endurance of marathon running, sports challenge individuals to push their physical and mental limits. Beyond fitness, they teach valuable life skills such as leadership, perseverance, and handling both victory and defeat with grace."

# Math:
math = "Mathematics is the language of patterns and logic that underpins much of our understanding of the world. It provides the foundation for disciplines ranging from physics and engineering to economics and computer science. Through abstraction and reasoning, math allows us to model natural phenomena, solve complex problems, and make predictions with precision. Far from being purely theoretical, mathematics powers everyday technologies like GPS, data encryption, and artificial intelligence. Its beauty lies in the balance between elegant simplicity and immense power to describe reality."

# Politics:
politics = "Politics is the process by which societies make collective decisions and distribute power and resources. It encompasses debates over governance, justice, and public policy, shaping the rules that guide communities and nations. While often contentious, politics reflects the diverse interests and values of people living together in a shared system. Through participation—voting, activism, and dialogue—citizens influence how societies evolve and address challenges such as inequality, climate change, and technological change. At its best, politics aspires to balance competing needs and uphold the principles of fairness and democracy."

**Problem 2:** Form a new dictionary and compute the probabilities each word appears in each topic.  Print the word and the probabilities in python position 9.

In [10]:
topics = {"Sports": sports, "Math": math, "Politics": politics}
topic_vocab = make_dictionary(*topics.values())
topic_probs = {
    name: word_probabilities(text, topic_vocab)
    for name, text in topics.items()
}

index = 9
print(topic_vocab[index])
for name, probs in topic_probs.items():
    print(f"{name}: {probs[index]:.6f}")

as
Sports: 0.012658
Math: 0.000000
Politics: 0.011494


##Part II: Determining the topic

Now we want to provide new text and determine the likelihood the sentence came from each of the three topics.  That is, our new sentence is like a region and we want to integrate the probability "density" we created over this region.

**Problem 3:** Compute the likelihood $L$ by summing the probability each word in the sentence appears in the topic.  Print the likelihood the sentence came from each topic.

In [11]:
sent1 = "The soccer team's victory is due to their teamwork, fitness, and discipline."


def topic_scores(sentence):
    counts = count_vector(sentence, topic_vocab)
    return {name: counts @ probs for name, probs in topic_probs.items()}

scores = topic_scores(sent1)
for name, score in scores.items():
    print(f"{name}: {score:.6f}")

Sports: 0.253165
Math: 0.180723
Politics: 0.183908


**Problem 4:** What topic is the sentence about.  Why?

The sentence is about **Sports**, which gets the highest score (0.253165). Words such as `soccer`, `victory`, `teamwork`, and `fitness` all contribute to the sports score.

**Problem 5:** What topic is the following sentence about?

The sentence is about **Math**, which gets the highest score (0.325301). It shares words such as `mathematics`, `patterns`, and `logic` with the math paragraph.

In [12]:
sent2 = "I love mathematics because of the patterns and logic that can be used to understand the physical world."

# your code here

scores = topic_scores(sent2)
for name, score in scores.items():
    print(f"{name}: {score:.6f}")
print("Predicted topic:", max(scores, key=scores.get))

Sports: 0.278481
Math: 0.325301
Politics: 0.252874
Predicted topic: Math


**Problem 6:** Write a sentence that fools the computer.  That is, write about one topic that the computer identifies as coming from another topic.

The sentence describes a mayor vetoing a bill, a political action. The classifier picks **Math** because the subject of the bill adds several words from the math paragraph.

In [13]:
trick = (
    "The mayor vetoed a bill that would fund research in mathematics, "
    "physics, engineering, and computer science."
)
scores = topic_scores(trick)
for name, score in scores.items():
    print(f"{name}: {score:.6f}")
print("Predicted topic:", max(scores, key=scores.get))

Sports: 0.177215
Math: 0.228916
Politics: 0.195402
Predicted topic: Math


##Part III: Generating sentences

Here is a really simple version of ChatGPT.  We simply generate a string of words about a topic based on the probabilities those words appear.

First, a simple example of how to pick a random number based on a probability vector:

In [14]:
# here is a simple probability vector
p = [0.1, 0.2, 0.7]

# here is code to sample from the elements, 0, 1, 2 with the given probabilities
c = np.cumsum(p)
print(c)

r = np.random.rand()
print(r)
for k in range(len(c)):
  if r < c[k]:
    print(k)
    break

[0.1 0.3 1. ]
0.7063968779106276
2


**Problem 6:** Write code that creates a random 15 word "sentence" for each topic.

In [15]:
rng = np.random.default_rng(235)

for name, probs in topic_probs.items():
    words = rng.choice(topic_vocab, size=15, replace=True, p=probs)
    print(name, " ".join(words))

Sports mental and and and they generations the goals professional such skills people vital both teach
Math mathematics everyday to much and to elegant and and and model logic economics data natural
Politics make decisions to competing system rules collective balance politics politics change through living the and


As you see, you get nonsense because there is no gramatical structure.  One improvement is to use bigrams (sets to two words) instead of individual words.  Starting from a random word, the next word is picked randomly from pairs containing that starting word.  Then the process is repeated from the new word.  This is one step closer to how ChatGPT generates text; ChatGPT generated this code.

In [16]:
import nltk
from nltk.corpus import reuters   # we need a much larger set of words to get this method to work
from collections import defaultdict
import random

# Make sure you have the corpus downloaded:
ensure_nltk_resource('corpora/reuters', 'reuters')

# 1. Load and tokenize sentences from the Reuters corpus
sentences = reuters.sents()
print(f"Number of sentences: {len(sentences)}")

Number of sentences: 54716


In [17]:
# 2. Build a bigram model: P(next_word | current_word)
model = defaultdict(list)
for sent in sentences:
    for w1, w2 in nltk.bigrams(sent):
        model[w1].append(w2)

# 3. Generate a sentence by random walk
def generate_sentence(model, start_word=None, length=15):
    if start_word is None:
        start_word = random.choice(list(model.keys()))
    word = start_word
    sentence = [word]
    for _ in range(length - 1):
        if word not in model or not model[word]:
            break
        word = random.choice(model[word])
        sentence.append(word)
    return ' '.join(sentence)

**Problem 7:** Generate a bunch of sentences, copy and paste three that are reasonable and three that are nonsense

In [18]:
random.seed(235)

generated_sentences = []
# Try a common sentence starter and keep the first sentence it generates.
for _ in range(40):
    text = generate_sentence(model, start_word="He", length=10)
    first_sentence, period, _ = text.partition(" . ")
    generated_sentences.append(first_sentence + "." if period else text)

generated_sentences.extend(generate_sentence(model, length=15) for _ in range(20))
for sentence in generated_sentences:
    print(sentence)

He said its direct cash.
He also said in 1986 , which last week .
He continues to buy back from 9 mln cubic metres
He added.
He said.
He also said the surplus , they said.
He said in Glasgow , it agreed at the fall
He would contribute to talks.
He said Darwin Beck of local unit to difficulties meant
He did not include 20 billion dlrs a share for
He told Reuters.
He said Mark IV & lt ; RNI > TO
He was also subject to Commerce Secretary said exact form
He acknowledged that the settlement of the " We very
He said.
He said there might appear to increase to take control
He also agreed to use American Forest Industries Inc unit
He said not succeed , said this week , control
He Kang , it now holds 5.
He also said.
He estimated 1.
He attributed the equivalent price for our board.
He said the product by buyers.
He said.
He said attempts to 88 billion Canadian oil industry commissioner
He added.
He noted , Reebok announcement by 13.
He said that the U.
He said its credit equivalent but noncommit

### Your good sentences:

- He would contribute to talks.
- He also agreed to use American Forest Industries Inc unit
- He would be followed.

### Your bad sentences:

- He and gasoline ?"
- hardwood lumber mills are in non - jung , 000 ounces in parts makers ,
- coated and fruit juice by nine NOTE : Mexico after Brazil ' S . S